# Chapter 2 – Command Anatomy, Documentation, and AI Assistance

This notebook is both:
- The **slide deck** for a ~1 hour in-class session (use Jupyter/Colab slideshow mode).
- A **guided homework notebook** (~6 hours) where you interact with an AI coding assistant to build a tiny package manager and related tools.

You will:
- Design a minimal package format for our custom shell ecosystem.
- Define a standard **anatomy** for command-line apps that unifies CLI, docs, shell integration, and packaging.
- Use an AI assistant to implement a local package manager in C.
- Extend it with an online registry backed by a small Node.js server.
- Refactor argument parsing and `--help` using `argtable3` and the app anatomy.
- (Optional) Design a natural-language interface for the shell using an LLM and a special `@` prefix.

> Implementation note: The file `CLItools.md` in the repository root lists the **required shell commands and options** for the full environment (including agent/MCP integration expectations like `--json`). When designing or refactoring commands with the app anatomy (`cmd_spec_t` + `argtable3`), refer to that list for names and flags.


## How to use this notebook

- **Instructor:** mark high-level Markdown cells as slides and hide detailed instructions as notes, if desired.
- **Students:** work through the notebook top to bottom. When you see a **Prompt to AI**, copy/adapt it into your AI tool and iterate.
- This notebook assumes you are running in **Google Colab** or a similar Linux environment.
- You can mount your Google Drive to store source files and test builds.

> Tip: Treat the AI as a smart collaborator. You stay in control of architecture and decisions; use the AI for boilerplate, refactoring, and explanations.


---

## Anatomy of a command-line app in this course

In this course, we treat each Unix-style command (e.g. `ls`, `wc`, `cat`, `pkg`) as a **module** with a standard *anatomy*.

The same module can be:
- A **built-in** command in your custom shell.
- A **standalone binary** (`ls`, `wc`, etc.).
- A source of **documentation** (help text, README) for packages.

To make this work, we define a common structure `cmd_spec_t` and require each command to provide:
- Command metadata (name, short and long description).
- Argument/option definitions (`argtable3`).
- A single `run` function that implements the logic.
- A `print_usage` function that prints help based on the same `argtable3` definitions.

This section defines that anatomy. Part 3 will show how `argtable3` plugs into it and how the package manager uses it for documentation.


### `cmd_spec_t`: command specification

We use a small struct to describe each command:

```c
typedef struct cmd_spec {
    const char *name;        // command name, e.g. "ls"
    const char *summary;     // one-line description
    const char *long_help;   // longer description / Markdown (may be NULL)

    // Main entrypoint: parses args (using argtable3) and runs the command.
    int (*run)(int argc, char **argv);

    // Prints usage and option help (using argtable3) to the given stream.
    void (*print_usage)(FILE *out);
} cmd_spec_t;
```

Each command module must define **exactly one** `cmd_spec_t`, for example for `hello`:

```c
int hello_run(int argc, char **argv);
void hello_print_usage(FILE *out);

cmd_spec_t cmd_hello_spec = {
    .name        = "hello",
    .summary     = "print a friendly greeting",
    .long_help   = "Print a greeting, optionally addressing a specific NAME.",
    .run         = hello_run,
    .print_usage = hello_print_usage,
};
```

Later, the shell will register `cmd_hello_spec` as a built-in, and the `pkg` tool will use `summary` / `long_help` when generating package metadata and docs.


### Example: `appl/hello` module with all parts

This repository includes a small, concrete example of the app anatomy in:

- `apps/hello/cmd_spec.h` – defines `cmd_spec_t` and the registry API.
- `apps/hello/cmd_hello.c` – implements the `hello` command module:
  - `hello_run` and `hello_print_usage` using `argtable3`.
  - `cmd_hello_spec` and `register_hello_command`.
- `apps/hello/hello_main.c` – a tiny `main()` that calls `cmd_hello_spec.run`.
- `apps/hello/registry.c` – a minimal in-memory registry implementation.
- `apps/hello/Makefile` – builds both:
  - `hello` standalone binary.
  - `libhello.a` library (for linking into a shell).

You can open these files and compare them to the anatomy described here. They form a complete, minimal example that you (or an AI assistant) can copy when refactoring existing commands like `ls` or `wc`.


### APPANATOMY.md – helping the AI refactor existing commands

To make refactoring easier, this repository contains a file:

- `PackageManagement/APPANATOMY.md`

It is **written for an AI assistant** and describes in detail:
- The `cmd_spec_t` type and required fields.
- How to use `argtable3` inside `run()` and `print_usage()`.
- How to integrate with the command registry.
- How to split an existing `main()` into `run()`, `print_usage()`, and a small wrapper.

When you ask an AI to refactor an existing command (e.g. your current `ls.c` or `wc.c`) into the new anatomy, you should:

1. Show the AI the contents of `APPANATOMY.md`.
2. Paste the current implementation of your command.
3. Ask it to refactor that code into the standard module form (`cmd_spec_t`, `run`, `print_usage`, etc.) without changing behavior.

Later in this notebook we will give concrete prompts for this workflow.


### Guidelines for working with the AI

When using the AI assistant:

- **Set the context clearly** in your first message:
  - What you are building (a `pkg` tool in C).
  - Your environment (Linux, GCC, access to `tar`, possibly `curl`).
  - Constraints (no external heavy libraries; keep it portable).
- **Break the task into small steps**:
  - First a skeleton, then one subcommand at a time.
- **Ask for compilable code** and **build instructions**:
  - e.g., "Show the `gcc` command to compile this".
- **Review and test**:
  - Read the generated code.
  - Compile and run it.
  - Ask the AI to help debug any warnings or runtime errors.

> Important: Do not paste your entire project if it is huge. Share only the relevant files or snippets (e.g., your current `pkg.c` or shell main loop).


---

## `argtable3`, command registry, and docs from the app anatomy

In Part 0 we defined the **anatomy** of a command (`cmd_spec_t`) and the shell’s registry.

In this part, we:
- Use **`argtable3`** inside each command module to define options and arguments.
- Make `argtable3` the **single source of truth** for:
  - CLI parsing.
  - `--help` behavior.
  - Documentation text that ends up in packages.
- Show how to refactor existing commands using the anatomy and `APPANATOMY.md`.

The key idea: once a command is written in this standard pattern, we can **automatically derive package metadata and docs** from it.


### `argtable3` inside a command module

Recall that each command module provides:

- `int <name>_run(int argc, char **argv);`
- `void <name>_print_usage(FILE *out);`
- `cmd_spec_t cmd_<name>_spec;`

Both `run` and `print_usage` should use the **same `argtable3` definitions**. Conceptually:

```c
#include "argtable3.h"

static void build_ls_argtable(struct arg_lit **help,
                              struct arg_lit **all,
                              struct arg_file **paths,
                              struct arg_end **end,
                              void ***argtable_out)
{
    *help  = arg_lit0("h", "help", "show help and exit");
    *all   = arg_lit0("a", "all",  "do not ignore entries starting with .");
    *paths = arg_file0(NULL, NULL, "[PATH...]", "directories or files to list");
    *end   = arg_end(20);

    static void *argtable[5];
    argtable[0] = *help;
    argtable[1] = *all;
    argtable[2] = *paths;
    argtable[3] = *end;
    argtable[4] = NULL;

    *argtable_out = argtable;
}
```

`run` uses this to parse arguments:

```c
int ls_run(int argc, char **argv)
{
    struct arg_lit  *help;
    struct arg_lit  *all;
    struct arg_file *paths;
    struct arg_end  *end;
    void           **argtable;

    build_ls_argtable(&help, &all, &paths, &end, &argtable);

    int nerrors = arg_parse(argc, argv, argtable);

    if (help->count > 0) {
        ls_print_usage(stdout);
        return 0;
    }
    if (nerrors > 0) {
        arg_print_errors(stdout, end, "ls");
        ls_print_usage(stdout);
        return 1;
    }

    // Existing ls logic goes here, using values from `all`, `paths`, etc.
}
```

`print_usage` uses the same builder:

```c
void ls_print_usage(FILE *out)
{
    struct arg_lit  *help;
    struct arg_lit  *all;
    struct arg_file *paths;
    struct arg_end  *end;
    void           **argtable;

    build_ls_argtable(&help, &all, &paths, &end, &argtable);

    fprintf(out, "Usage: ls ");
    arg_print_syntax(out, argtable, "
");
    fprintf(out, "
Options:
");
    arg_print_glossary(out, argtable, "  %-20s %s
");
}
```

> The exact option set is up to you and your existing commands. The important part is that **all parsing and help** comes from one `argtable3` definition per command.


### Documentation from `argtable3` and `cmd_spec_t`

Now we can generate documentation automatically:

- `cmd_spec_t.summary` and `cmd_spec_t.long_help` provide high-level text.
- `print_usage()` provides detailed syntax and option descriptions.

There are two practical ways to use this for packages:

1. **Direct call from the package manager** (when `pkg` links the command libraries):
   - `pkg` loads all `cmd_spec_t` via the registry.
   - It calls `spec->print_usage()` and captures the output into a documentation file (e.g., `docs/<name>-usage.txt`).
   - It writes `pkg.json` with `description` taken from `spec->summary`.

2. **Special help mode from the standalone binary**:
   - Each command supports a flag like `--help-md` or `--help-json` that prints Markdown/JSON help based on argtable.
   - `pkg build` runs the command in that mode and captures the output to include in the package.

Either way, the **source of truth** stays in the command module, and `pkg` just asks the command to describe itself.


---

## Summary and next steps

In this notebook, you:

- Defined a reusable **anatomy** for command-line apps (`cmd_spec_t`, registry, `argtable3`-based parsing).


Suggested next steps:
- Integrate `argtable3` into your shell’s built-in commands and ensure `help` uses the registry and `print_usage`.
